In [1]:
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
import numpy as np

### ! This script reads in the water mask generated from the S2_watermask script in GEE.

## Check CRS

In [2]:
%%bash
gdalinfo /Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI.tif

Driver: GTiff/GeoTIFF
Files: /Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI.tif
Size is 13637, 4625
Coordinate System is:
GEOGCRS["WGS 84",
    ENSEMBLE["World Geodetic System 1984 ensemble",
        MEMBER["World Geodetic System 1984 (Transit)"],
        MEMBER["World Geodetic System 1984 (G730)"],
        MEMBER["World Geodetic System 1984 (G873)"],
        MEMBER["World Geodetic System 1984 (G1150)"],
        MEMBER["World Geodetic System 1984 (G1674)"],
        MEMBER["World Geodetic System 1984 (G1762)"],
        MEMBER["World Geodetic System 1984 (G2139)"],
        MEMBER["World Geodetic System 1984 (G2296)"],
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]],
        ENSEMBLEACCURACY[2.0]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
    CS[ellipsoidal,2],
        AXIS["geodetic latitude (Lat)",north,
            ORDER[1],
            ANGLEUNIT["degree",0.0174532925199433]],

In [3]:
%%bash
gdalwarp -r bilinear -overwrite \
  -s_srs EPSG:4326 \
  -t_srs EPSG:4326 \
  /Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI.tif \
  /Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI_EPSG4326.tif

Creating output file that is 13637P x 4625L.
Processing /Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.


# Vectorize raster

In [ ]:
water_mask_in_path = "/Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI_EPSG4326.tif"
water_mask_out_path = "/Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI_EPSG4326_polygon.shp"

# Read in the water mask
with rasterio.open(water_mask_in_path) as water_mask:
    array = water_mask.read(1)
    transform = water_mask.transform
    crs = water_mask.crs

# confirm CRS
print("CRS:", water_mask.crs)

# Mask for water pixels (1 = water, 0 = nah not water)
mask = (array == 1)

# Polygonize!
polys = []
for geom, val in shapes(array, mask=mask, transform=transform):
    polys.append({"geometry": shape(geom), "value": int(val)})

# Make the polygons a GeoDataFrame
gdf = gpd.GeoDataFrame(polys, crs=crs)

# Filter out tiny polygons
gdf = gdf[gdf.geometry.area > 100]

# Save as a shapefile
gdf.to_file(water_mask_out_path, driver="ESRI Shapefile")
print("Shapefile saved to:", water_mask_out_path)


Shapefile saved to: /Users/camryn/Documents/UNC/Ice_caval/Tanana/SWORD_clipped_TN_domain/S2_WaterMask_NDWI_EPSG4326_polygon.shp
